# Evaluate — DDPG A1 & A2, TD3 A1 & A2 on Juan's 4 days

Loads the four CURRENT best policies (`policies/*_best.pt`, trained in the tuned
stability-safe gain box `GAIN_LOW=[-30,-4.0,-0.4]`) and evaluates them on the four
newest Juan days: metrics table (MAE / RMSE / peak overshoot / undershoot) vs the
expert PI, one tracking chart per day (all four policies + expert), and the applied
flow. Charts are saved under `charts/evaluate/`. Evaluation only — no training.


In [ ]:
# ── Setup: shared library (../main_script) + this folder's config ────────────
import os, sys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
sys.path.insert(0, os.getcwd())                                            # for: import config
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), os.pardir)))  # reach ../main_script
from main_script import *
import config as cfg
from config import *
configure(cfg)

# ══ ONE SWITCH: which generation to evaluate ══════════════════════════════════
# False -> TUNED policies (policies/*_best.pt, wide stability-safe box)
# True  -> OLD-window policies (policies/old_box/*_best_oldbox.pt, narrow box)
# The gain box, policy paths, chart filenames and CSV all flip TOGETHER, so the
# two generations can never be mixed or overwrite each other's outputs.
USE_OLD_BOX = False

# ══ box0.4/0.8/1.0/1.2/1.4 SWEEP CANDIDATE ════════════════════════════════════
# None -> ignored (USE_OLD_BOX picks old-window vs tuned as above).
# 'box0.4'/'box0.8'/'box1.0'/'box1.2'/'box1.4' -> which margin-swept generation
# to evaluate. ALL_BOX_CANDIDATE controls how many policies actually switch:
#   ALL_BOX_CANDIDATE=False -> ONLY TD3-A2 swaps, to its checkpoint from
#     _sweep_train_td3a2.py, decoded through its own box; DDPG-A1/A2 and TD3-A1
#     stay on the tuned box they were trained with (matches _sweep_screen.py).
#   ALL_BOX_CANDIDATE=True  -> ALL FOUR swap to their _sweep_train_baselines_box.py
#     / _sweep_train_td3a2.py checkpoints, all decoded through the candidate box
#     (needed for a fair 4-way comparison -- decoding a tuned-box-trained actor
#     through a different box's LOW/HIGH does NOT retrain it, it just remaps the
#     same raw output to different physical gains).
BOX_CANDIDATE = None       # e.g. 'box1.2'
ALL_BOX_CANDIDATE = False  # True once DDPG-A1/A2 + TD3-A1 have box-suffixed checkpoints too

OLD_LOW   = np.array([-6.0, -0.100, -0.60], dtype=np.float32)
TUNED_LOW = np.array([-30.0, -4.0, -0.40], dtype=np.float32)
GAIN_HIGH_FIXED = np.array([-0.05, -0.0001, 0.10], dtype=np.float32)
CANDIDATE_T = {'box0.4': 0.1546, 'box0.8': 0.3633, 'box1.0': 0.4839,
               'box1.2': 0.6210, 'box1.4': 0.7845}

if USE_OLD_BOX:
    GEN = 'old-window'; SUFFIX = '_oldbox'
    POLICIES = {
        'DDPG-A1': os.path.join('old_box', 'ddpg_online_ac_approach1_best_oldbox.pt'),
        'DDPG-A2': os.path.join('old_box', 'ddpg_online_ac_approach2_best_oldbox.pt'),
        'TD3-A1':  os.path.join('old_box', 'td3_online_ac_approach1_best_oldbox.pt'),
        'TD3-A2':  os.path.join('old_box', 'td3_online_ac_approach2_best_oldbox.pt'),
    }
    BOXES = {tag: (OLD_LOW, GAIN_HIGH_FIXED) for tag in POLICIES}
elif BOX_CANDIDATE is not None and ALL_BOX_CANDIDATE:
    t = CANDIDATE_T[BOX_CANDIDATE]
    box_low = (OLD_LOW + t * (TUNED_LOW - OLD_LOW)).astype(np.float32)
    GEN = f'{BOX_CANDIDATE}-all'; SUFFIX = f'_{BOX_CANDIDATE}all'
    POLICIES = {
        'DDPG-A1': f'ddpg_online_ac_approach1_{BOX_CANDIDATE}_best.pt',
        'DDPG-A2': f'ddpg_online_ac_approach2_{BOX_CANDIDATE}_best.pt',
        'TD3-A1':  f'td3_online_ac_approach1_{BOX_CANDIDATE}_best.pt',
        'TD3-A2':  f'td3_online_ac_approach2_{BOX_CANDIDATE}_best.pt',
    }
    BOXES = {tag: (box_low, GAIN_HIGH_FIXED) for tag in POLICIES}
elif BOX_CANDIDATE is not None:
    t = CANDIDATE_T[BOX_CANDIDATE]
    box_low = (OLD_LOW + t * (TUNED_LOW - OLD_LOW)).astype(np.float32)
    GEN = f'tuned+{BOX_CANDIDATE}'; SUFFIX = f'_{BOX_CANDIDATE}'
    POLICIES = {
        'DDPG-A1': 'ddpg_online_ac_approach1_best.pt',
        'DDPG-A2': 'ddpg_online_ac_approach2_best.pt',
        'TD3-A1':  'td3_online_ac_approach1_best.pt',
        'TD3-A2':  f'td3_online_ac_approach2_{BOX_CANDIDATE}_best.pt',
    }
    BOXES = {'DDPG-A1': (TUNED_LOW, GAIN_HIGH_FIXED), 'DDPG-A2': (TUNED_LOW, GAIN_HIGH_FIXED),
             'TD3-A1':  (TUNED_LOW, GAIN_HIGH_FIXED), 'TD3-A2':  (box_low,   GAIN_HIGH_FIXED)}
else:
    GEN = 'tuned'; SUFFIX = ''
    POLICIES = {
        'DDPG-A1': 'ddpg_online_ac_approach1_best.pt',
        'DDPG-A2': 'ddpg_online_ac_approach2_best.pt',
        'TD3-A1':  'td3_online_ac_approach1_best.pt',
        'TD3-A2':  'td3_online_ac_approach2_best.pt',
    }
    BOXES = {tag: (TUNED_LOW, GAIN_HIGH_FIXED) for tag in POLICIES}

cfg.GAIN_LOW, cfg.GAIN_HIGH = next(iter(BOXES.values()))  # placeholder; set per-actor at each rollout call

# ══ FLOW-RATE WINDOW (pump limits) ════════════════════════════════════════════
# Default (what every policy was TRAINED with): Q_MIN=0.0, Q_MAX=40.0 L/min.
# Change the two numbers below to evaluate the same policies under different
# actuator limits (a what-if robustness test): the env clips the commanded flow
# to this window, so it affects saturation, overshoot and the anti-windup path.
# When the window differs from the training default, chart/CSV filenames get a
# '_qMIN-MAX' suffix automatically so default outputs are never overwritten.
cfg.Q_MIN = 0.0        # lower pump limit [L/min]
cfg.Q_MAX = 40.0       # upper pump limit [L/min]
Q_MIN, Q_MAX = float(cfg.Q_MIN), float(cfg.Q_MAX)   # re-bind names used by the chart cells
if (Q_MIN, Q_MAX) != (0.0, 40.0):
    SUFFIX += f'_q{Q_MIN:g}-{Q_MAX:g}'

# ══ SETPOINT (T_ref) ══════════════════════════════════════════════════════════
# Default: sunny days (all 4 Juan days) track TREF_SUNNY = 80.0 C.
# Set NEW_TREF to a number (e.g. 75.0) to evaluate every day at that setpoint
# instead; None keeps the default. This flows through EVERYTHING consistently
# (control error, the T_ref element of the policy's state, metrics, charts).
# NOTE: the observation box covers T_ref in [55, 85] C and the policies only ever
# saw 65/80 C in training - setpoints outside 55-85 extrapolate the network and
# results far from 65/80 are a zero-shot generalisation test, not tuned behaviour.
# Outputs get a '_tref<value>' suffix automatically when non-default.
NEW_TREF = None        # e.g. 75.0, or None for the 80.0 C default

if NEW_TREF is not None:
    cfg.TREF_SUNNY = float(NEW_TREF)
    SUFFIX += f'_tref{float(NEW_TREF):g}'

COLORS = {'DDPG-A1': '#d62728', 'DDPG-A2': '#ff7f0e', 'TD3-A1': '#1f77b4', 'TD3-A2': '#9467bd'}
actors = {tag: load_actor_raw(os.path.join(SAVE_DIR, f)) for tag, f in POLICIES.items()}

EV_DIR = os.path.join(CHART_DIR, 'evaluate'); os.makedirs(EV_DIR, exist_ok=True)

def undershoot_metric(T, tref):
    """Worst dip below T_ref AFTER the first crossing (startup climb excluded)."""
    above = np.nonzero(T >= tref)[0]
    return float(np.max(tref - T[above[0]:])) if len(above) else float(np.max(tref - T))

for tag, (lo, hi) in BOXES.items():
    print(f"[{GEN}] {tag} decode box LOW={list(lo)} HIGH={list(hi)}")
print(f"flow window: q in [{Q_MIN:g}, {Q_MAX:g}] L/min" +
      ("  (NON-DEFAULT)" if (Q_MIN, Q_MAX) != (0.0, 40.0) else "  (training default)"))
print(f"setpoint: T_ref = {cfg.TREF_SUNNY:g} C" +
      ("  (NON-DEFAULT - outputs get suffix '" + SUFFIX + "')" if NEW_TREF is not None else "  (training default 80.0)"))


In [ ]:
# ── STATE vs time: metrics table + one chart per day (Tout + applied flow) ───
days = [load_dataset(f) for f in JUAN_FILES]

rows = []
for d in days:
    tr = dataset_tref(d['name'])
    Te, Qe = rollout_expert(d)
    m, rm = mae_rmse(Te, tr)
    rows.append(dict(day=d['name'][:11], policy='Expert', MAE=m, RMSE=rm,
                     ovr=peak_overshoot(Te, tr), und=undershoot_metric(Te, tr)))
    Ts = {}; Qs = {}
    for tag, actor in actors.items():
        cfg.GAIN_LOW, cfg.GAIN_HIGH = BOXES[tag]
        T, Q, G = rollout_full(actor, d); Ts[tag] = T; Qs[tag] = Q
        m, rm = mae_rmse(T, tr)
        rows.append(dict(day=d['name'][:11], policy=tag, MAE=m, RMSE=rm,
                         ovr=peak_overshoot(T, tr), und=undershoot_metric(T, tr)))

    # chart: Tout (top, all policies + expert) and applied flow q (bottom)
    t = np.arange(len(Te))
    fig, ax = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    ax[0].axhline(tr, ls='--', c='green', lw=1.2, label=f'T_ref = {tr:.0f} C')
    ax[0].plot(t, Te, c='#888888', ls='-.', lw=0.9, label=f'Expert  MAE={rows[-5]["MAE"]:.3f}')
    for tag in POLICIES:
        mt = [r for r in rows if r['day'] == d['name'][:11] and r['policy'] == tag][0]
        ax[0].plot(np.arange(len(Ts[tag])), Ts[tag], c=COLORS[tag], lw=1.0,
                   label=f"{tag}  MAE={mt['MAE']:.3f} ovr={mt['ovr']:.2f} und={mt['und']:.2f}")
    ax[0].set_ylabel('Tout (C)'); ax[0].grid(alpha=.3); ax[0].legend(fontsize=8, ncol=2)
    ax[1].plot(t, Qe, c='#888888', ls='-.', lw=0.9, label='q Expert')
    for tag in POLICIES:
        ax[1].plot(np.arange(len(Qs[tag])), Qs[tag], c=COLORS[tag], lw=0.8, label=f'q {tag}')
    ax[1].axhline(Q_MIN, color='gray', ls=':'); ax[1].axhline(Q_MAX, color='gray', ls=':')
    ax[1].set_ylabel('Flow q (L/min)'); ax[1].set_xlabel('time step'); ax[1].grid(alpha=.3)
    ax[1].legend(fontsize=8, ncol=2)
    fig.tight_layout()
    fig.savefig(os.path.join(EV_DIR, f"evaluate_day_{d['name'][:11]}{SUFFIX}.png"),
                dpi=140, bbox_inches='tight')
    plt.show(); plt.close(fig)

df = pd.DataFrame(rows)
print('Per-day metrics:')
display(df.round(3))
summ = df.groupby('policy').agg(MAE=('MAE', 'mean'), RMSE=('RMSE', 'mean'),
                                ovr_worst=('ovr', 'max'), und_worst=('und', 'max'))
summ = summ.loc[['Expert', 'DDPG-A1', 'DDPG-A2', 'TD3-A1', 'TD3-A2']]
print(f'Summary [{GEN}] over the 4 Juan days (mean MAE/RMSE; worst-day ovr/und):')
display(summ.round(3))
df.to_csv(os.path.join(EV_DIR, f'evaluate_juan_days{SUFFIX}.csv'), index=False)


In [ ]:
# ── ACTION vs time (with Expert PI): one chart per day (Kp, Ki, Kw + flow q) ──
# Same style as the state chart above: all four policies + the expert reference.
# The gains are the actor's ACTIONS decoded through the active gain box.
EXP_G = {'Kp': KP_EXPERT, 'Ki': KI_EXPERT, 'Kw': KI_EXPERT / KP_EXPERT}
ACT_ROWS = [('Kp', 'Kp  [-]'), ('Ki', 'Ki  [-]'), ('Kw', 'Kw (anti-windup)  [-]'),
            ('q',  'flow q  [L/min]')]

ACT_G = {}; ACT_Q = {}          # cached rollouts, reused by the RL-only cell below
for d in days:
    tr = dataset_tref(d['name'])
    Te, Qe = rollout_expert(d)
    Gs = {}; Qs = {}
    for tag, actor in actors.items():
        cfg.GAIN_LOW, cfg.GAIN_HIGH = BOXES[tag]
        T, Q, G = rollout_full(actor, d)
        Gs[tag] = G; Qs[tag] = Q
    ACT_G[d['name']] = Gs; ACT_Q[d['name']] = Qs

    fig, ax = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
    for r, (key, ylab) in enumerate(ACT_ROWS):
        if key == 'q':
            t = np.arange(len(Qe))
            ax[r].plot(t, Qe, c='#888888', ls='-.', lw=0.9, label='Expert PI')
            for tag in POLICIES:
                ax[r].plot(np.arange(len(Qs[tag])), Qs[tag], c=COLORS[tag], lw=0.8, label=tag)
            ax[r].axhline(Q_MIN, color='gray', ls=':'); ax[r].axhline(Q_MAX, color='gray', ls=':')
            ax[r].set_xlabel('time step')
        else:
            gi = {'Kp': 0, 'Ki': 1, 'Kw': 2}[key]
            ax[r].axhline(EXP_G[key], ls='-.', c='#888888', lw=0.9,
                          label=f'Expert ({EXP_G[key]:.4f})')
            for tag in POLICIES:
                ax[r].plot(np.arange(len(Gs[tag])), Gs[tag][:, gi], c=COLORS[tag], lw=0.8, label=tag)
        ax[r].set_ylabel(ylab); ax[r].grid(alpha=.3)
        if r == 0:
            ax[r].legend(fontsize=8, ncol=3)
    fig.tight_layout()
    fig.savefig(os.path.join(EV_DIR, f"evaluate_actions_day_{d['name'][:11]}{SUFFIX}.png"),
                dpi=140, bbox_inches='tight')
    plt.show(); plt.close(fig)


In [ ]:
# ── ACTION vs time (RL policies ONLY, no Expert PI) ──────────────────────────
# The expert's constant gains (Kp=-0.5, Ki=-0.0017) are on a far smaller scale
# than the RL gains, so this version omits them to keep the RL actions readable.
# Reuses the rollouts cached by the cell above; saves to *_rlonly_* filenames so
# the original charts (with the expert reference) are never overwritten.
for d in days:
    Gs = ACT_G[d['name']]; Qs = ACT_Q[d['name']]
    fig, ax = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
    for r, (key, ylab) in enumerate(ACT_ROWS):
        if key == 'q':
            for tag in POLICIES:
                ax[r].plot(np.arange(len(Qs[tag])), Qs[tag], c=COLORS[tag], lw=0.8, label=tag)
            ax[r].axhline(Q_MIN, color='gray', ls=':'); ax[r].axhline(Q_MAX, color='gray', ls=':')
            ax[r].set_xlabel('time step')
        else:
            gi = {'Kp': 0, 'Ki': 1, 'Kw': 2}[key]
            for tag in POLICIES:
                ax[r].plot(np.arange(len(Gs[tag])), Gs[tag][:, gi], c=COLORS[tag], lw=0.8, label=tag)
        ax[r].set_ylabel(ylab); ax[r].grid(alpha=.3)
        if r == 0:
            ax[r].legend(fontsize=8, ncol=4)
    fig.tight_layout()
    fig.savefig(os.path.join(EV_DIR, f"evaluate_actions_rlonly_day_{d['name'][:11]}{SUFFIX}.png"),
                dpi=140, bbox_inches='tight')
    plt.show(); plt.close(fig)


In [ ]:
# ── ACTION vs time, ONE PANEL PER METHOD: Kp/Ki/Kw for Expert + each policy ──
# Grid: 3 rows (Kp, Ki, Kw) x 5 columns (Expert, DDPG-A1, DDPG-A2, TD3-A1, TD3-A2).
# Each panel has its OWN y-scale, so the expert's small constant gains and the RL
# policies' large gains are both readable. Reuses the rollouts cached above.
GAIN_ROWS = [('Kp', 'Kp  [-]'), ('Ki', 'Ki  [-]'), ('Kw', 'Kw (anti-windup)  [-]')]
METHODS = ['Expert'] + list(POLICIES.keys())

for d in days:
    Gs = ACT_G[d['name']]
    N = min(len(G) for G in Gs.values())
    t = np.arange(N)
    fig, ax = plt.subplots(3, 5, figsize=(20, 9), sharex=True)
    for j, m in enumerate(METHODS):
        for r, (key, ylab) in enumerate(GAIN_ROWS):
            a = ax[r, j]
            if m == 'Expert':
                a.plot(t, np.full(N, EXP_G[key]), c='#888888', ls='-.', lw=1.2)
                a.annotate(f'{EXP_G[key]:.4f}', xy=(0.5, 0.5), xycoords='axes fraction',
                           ha='center', fontsize=9, color='#555555')
            else:
                gi = {'Kp': 0, 'Ki': 1, 'Kw': 2}[key]
                a.plot(t, Gs[m][:N, gi], c=COLORS[m], lw=0.9)
            if r == 0:
                a.set_title(m, fontsize=11)
            if j == 0:
                a.set_ylabel(ylab)
            if r == 2:
                a.set_xlabel('time step')
            a.grid(alpha=.3)
    fig.tight_layout()
    fig.savefig(os.path.join(EV_DIR, f"evaluate_actions_grid_day_{d['name'][:11]}{SUFFIX}.png"),
                dpi=140, bbox_inches='tight')
    plt.show(); plt.close(fig)
